# 📊 Day 21 — Random Forest
### 100 Days of Data Science | by Shoaib Aslam

---

## Table of Contents

1️⃣ What is Random Forest  
2️⃣ How It Works — Key Concepts  
3️⃣ Loading & Preparing the Data  
4️⃣ Random Forest Classifier  
5️⃣ Evaluating the Classifier  
6️⃣ Random Forest vs Decision Tree  
7️⃣ Hyperparameter Tuning  
8️⃣ Random Forest Regressor  
9️⃣ Feature Importance  
🔟 When to Use What  

---
## 1️⃣ What is Random Forest?

Random Forest is an **ensemble learning algorithm** that builds many Decision Trees and combines their predictions.

Instead of relying on one tree, it:
- Builds hundreds of trees on random subsets of data
- Each tree gives a prediction
- Final prediction = majority vote (classification) or average (regression)

> **Key idea:** A crowd of weak learners is smarter than one strong learner. That is the power of Random Forest.

---
## 2️⃣ How It Works — Key Concepts

| Term | Meaning |
|------|---------|
| Ensemble | Combining multiple models for better performance |
| Bagging | Each tree trains on a random sample of the data (with replacement) |
| Feature Randomness | Each tree only sees a random subset of features at each split |
| n_estimators | Number of trees in the forest |
| max_depth | Maximum depth of each tree |
| max_features | Number of features considered at each split |
| OOB Score | Out-of-bag score — built-in validation without a test set |

**Why it beats a single Decision Tree:**
- Less overfitting due to averaging
- More robust to noise and outliers
- Works well even without much hyperparameter tuning

---
## 3️⃣ Loading & Preparing the Data

In [ ]:
# Importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder

sns.set_style('whitegrid')
print("Libraries imported successfully!")

In [ ]:
# Load the Winemag dataset
df = pd.read_csv(r"C:\Users\aslam\Downloads\winemag-data-130k-v2.csv\winemag-data-130k-v2.csv", index_col=0)
df.head()

In [ ]:
# Keep relevant columns and drop missing values
df = df[['country', 'variety', 'points', 'price']].dropna()
print("Shape:", df.shape)

In [ ]:
# Create target variable — Premium (1) or Regular (0)
df['is_premium'] = (df['points'] >= 90).astype(int)

# Encode categorical columns
le_country = LabelEncoder()
le_variety = LabelEncoder()

df['country_encoded'] = le_country.fit_transform(df['country'])
df['variety_encoded'] = le_variety.fit_transform(df['variety'])

print("Premium wines:", df['is_premium'].sum())
print("Regular wines:", (df['is_premium'] == 0).sum())
df.head(3)

---
## 4️⃣ Random Forest Classifier

Predicting whether a wine is Premium or Regular.

In [ ]:
# Features and target
X = df[['price', 'country_encoded', 'variety_encoded']]
y = df['is_premium']

# Train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

In [ ]:
# Train Random Forest Classifier
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42, oob_score=True)
rf_clf.fit(X_train, y_train)

print("Model trained successfully!")
print("Number of Trees:", rf_clf.n_estimators)
print("OOB Score:", round(rf_clf.oob_score_, 4))

---
## 5️⃣ Evaluating the Classifier

In [ ]:
# Predictions
y_pred = rf_clf.predict(X_test)

# Accuracy
print("Accuracy:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")

In [ ]:
# Classification Report
print(classification_report(y_test, y_pred, target_names=['Regular', 'Premium']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Regular', 'Premium'],
            yticklabels=['Regular', 'Premium'])
plt.title("Confusion Matrix — Random Forest")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

---
## 6️⃣ Random Forest vs Decision Tree

Direct comparison — same data, same features, same split.

In [ ]:
# Train Decision Tree for comparison
dt_clf = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_clf.fit(X_train, y_train)
y_pred_dt = dt_clf.predict(X_test)

# Compare results
comparison = pd.DataFrame({
    'Model'    : ['Decision Tree (depth=5)', 'Random Forest (100 trees)'],
    'Accuracy' : [
        round(accuracy_score(y_test, y_pred_dt) * 100, 2),
        round(accuracy_score(y_test, y_pred) * 100, 2)
    ]
})
comparison

In [ ]:
# Accuracy vs Number of Trees
n_trees = [1, 5, 10, 20, 50, 100, 150, 200]
accuracies = []

for n in n_trees:
    clf = RandomForestClassifier(n_estimators=n, random_state=42)
    clf.fit(X_train, y_train)
    accuracies.append(accuracy_score(y_test, clf.predict(X_test)))

plt.figure(figsize=(12, 5))
plt.plot(n_trees, accuracies, marker='o', color='green', linewidth=2)
plt.title("Accuracy vs Number of Trees")
plt.xlabel("Number of Trees (n_estimators)")
plt.ylabel("Test Accuracy")
plt.grid(True)
plt.show()

---
## 7️⃣ Hyperparameter Tuning

Finding the best combination of hyperparameters using GridSearchCV.

In [ ]:
# Define parameter grid
param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth'   : [3, 5, 10, None],
    'max_features': ['sqrt', 'log2']
}

# GridSearchCV
grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best CV Accuracy:", round(grid_search.best_score_ * 100, 2), "%")

In [ ]:
# Train best model
best_rf = grid_search.best_estimator_
y_pred_best = best_rf.predict(X_test)

print("Best Model Test Accuracy:", round(accuracy_score(y_test, y_pred_best) * 100, 2), "%")

---
## 8️⃣ Random Forest Regressor

Random Forest can also predict numerical values.

In [ ]:
# Predict wine price
X_reg = df[['points', 'country_encoded', 'variety_encoded']]
y_reg = df['price']

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

In [ ]:
# Train Random Forest Regressor
rf_reg = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
rf_reg.fit(X_train_r, y_train_r)

y_pred_r = rf_reg.predict(X_test_r)

print("Random Forest Regressor Results:")
print(f"RMSE : {np.sqrt(mean_squared_error(y_test_r, y_pred_r)):.2f}")
print(f"R²   : {r2_score(y_test_r, y_pred_r):.4f}")

In [ ]:
# Actual vs Predicted plot
plt.figure(figsize=(8, 6))
plt.scatter(y_test_r, y_pred_r, alpha=0.3, color='coral')
plt.plot([y_test_r.min(), y_test_r.max()], [y_test_r.min(), y_test_r.max()], 'navy', linewidth=2, label='Perfect Prediction')
plt.title("Actual vs Predicted — Random Forest Regressor")
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.legend()
plt.show()

---
## 9️⃣ Feature Importance

In [ ]:
# Feature importance from classifier
feature_names = ['price', 'country_encoded', 'variety_encoded']
importances   = best_rf.feature_importances_

feat_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feat_df = feat_df.sort_values('Importance', ascending=False)

plt.figure(figsize=(8, 4))
sns.barplot(data=feat_df, x='Importance', y='Feature', palette='viridis')
plt.title("Feature Importance — Random Forest Classifier")
plt.xlabel("Importance Score")
plt.show()

In [ ]:
# Cross validation score
cv_scores = cross_val_score(best_rf, X, y, cv=5, scoring='accuracy')

print("Cross Validation Accuracy Scores:", cv_scores.round(4))
print("Mean Accuracy:", cv_scores.mean().round(4))
print("Std Dev:", cv_scores.std().round(4))

---
## 🔟 When to Use What

| Situation | What to Do |
|-----------|------------|
| Predict a category | RandomForestClassifier |
| Predict a number | RandomForestRegressor |
| Want built-in validation | Set `oob_score=True` |
| Need best hyperparameters | Use GridSearchCV |
| Want fast tuning | Use RandomizedSearchCV |
| Need feature importance | Use `.feature_importances_` |
| Model still overfitting | Reduce `max_depth` or increase `min_samples_split` |
| Need even better accuracy | Try XGBoost or Gradient Boosting |

---
## 🔑 Key Takeaways

→ Random Forest = many Decision Trees voting together  
→ It almost always outperforms a single Decision Tree  
→ Bagging and feature randomness are what make it powerful  
→ OOB score gives free validation without needing a separate test set  
→ More trees = better performance up to a point — then it plateaus  
→ GridSearchCV finds the best hyperparameters automatically  
→ Feature importance from Random Forest is more reliable than from a single tree  
→ Random Forest is the best starting point for most real-world ML problems

---
## 📁 Connect & Follow the Journey

| Platform | Link |
|----------|------|
| 🐙 GitHub | https://github.com/imshoaibaslam/100-Days-of-Data-Science |
| 📊 Kaggle | https://www.kaggle.com/shoaib32922 |

---
*Day 21 of 100 — Keep Learning, Keep Building 🚀*